In [2]:
!pip install --upgrade pip

In [3]:
!pip install torch numpy pandas cartopy matplotlib

In [4]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import torch.optim as optim

In [5]:
DATASET_DIR = '/cluster/tufts/c26sp1cs0137/data/assignment2_data/dataset'
metadata = torch.load(f'{DATASET_DIR}/metadata.pt', weights_only=False)


In [6]:
for key in metadata.keys():
    print("---------------------------------------------------------------------------")
    print(key)
    print(metadata[key])
len(metadata["values"])

---------------------------------------------------------------------------
variable_names
['TMP@2m_above_ground', 'RH@2m_above_ground', 'UGRD@10m_above_ground', 'VGRD@10m_above_ground', 'GUST@surface', 'DSWRF@surface', 'APCP_1hr_acc_fcst@surface', 'CAPE@surface', 'DPT@1000mb', 'DPT@500mb', 'DPT@700mb', 'DPT@850mb', 'DPT@925mb', 'HGT@1000mb', 'HGT@500mb', 'HGT@700mb', 'HGT@850mb', 'HGT@surface', 'TMP@1000mb', 'TMP@500mb', 'TMP@700mb', 'TMP@850mb', 'TMP@925mb', 'UGRD@1000mb', 'UGRD@250mb', 'UGRD@500mb', 'UGRD@700mb', 'UGRD@850mb', 'UGRD@925mb', 'VGRD@1000mb', 'VGRD@250mb', 'VGRD@500mb', 'VGRD@700mb', 'VGRD@850mb', 'VGRD@925mb', 'TCDC@entire_atmosphere', 'HCDC@high_cloud_layer', 'MCDC@middle_cloud_layer', 'LCDC@low_cloud_layer', 'PWAT@entire_atmosphere_single_layer', 'RHPW@entire_atmosphere', 'VIL@entire_atmosphere']
---------------------------------------------------------------------------
n_vars
42
---------------------------------------------------------------------------
input_shape

KeyError: 'values'

In [ ]:
targets = torch.load(f'{DATASET_DIR}/targets.pt', weights_only=False)

# for key in targets.keys():
#     print("---------------------------------------------------------------------------")
#     print(key)
#     print(targets[key])
# len(targets["values"])

In [ ]:
import torch
import torch.nn as nn

# ── 1. Convolution Layer ──────────────────────────────────────────────────────
class ConvLayer(nn.Module):
    def __init__(self, in_channels=42, out_channels=64):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1   # padding=1 keeps H/W identical with kernel_size=3
        )
        self.bn   = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (B, 42, 450, 449) → (B, 64, 450, 449)
        return self.relu(self.bn(self.conv(x)))


# ── 2. MaxPool Layer ──────────────────────────────────────────────────────────
class MaxPoolLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # x: (B, 64, 450, 449) → (B, 64, 225, 224)
        return self.pool(x)

# Linear layer
class FCLayer(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc      = nn.LazyLinear(num_classes)  # ← infers in_features automatically

    def forward(self, x):
        x = self.flatten(x)
        return self.fc(x)

In [ ]:
# # ── Quick sanity check ────────────────────────────────────────────────────────
# B = 2
# x = torch.randn(B, 42, 450, 449)

# conv  = ConvLayer()
# pool  = MaxPoolLayer()
# fc    = FCLayer()

# out1 = conv(x)
# print("After ConvLayer :", out1.shape)   # (2, 64, 450, 449)

# out2 = pool(out1)
# print("After MaxPoolLayer:", out2.shape) # (2, 64, 225, 224)

# out3 = fc(out2)
# print("After FCLayer    :", out3.shape)  # (2, 6)

In [ ]:
# Get the data for a specific year
# Total number of files per year
# 2018: 4128
# 2019: 8760
# 2020: 8784
# 2021: 4608

n_2018 = 4128
n_2019 = 8760
n_2020 = 8784
n_2021 = 4608

start_date_2018 = '2018-07-13T00:00'
start_date_2019 = '2019-01-01T00:00'
start_date_2020 = '2020-01-01T00:00'
start_date_2021 = '2021-01-01T00:00'

def generate_datetime_strings(start_date: str, n: int) -> list[str]:
    """
    Generate YYYYMMDDHH strings for n dates starting from start_date,
    incrementing by 24 hours each step.
    
    Args:
        start_date: Date string in format 'YYYY-MM-DDTHH:MM'
        n: Number of dates to generate
    
    Returns:
        List of strings in YYYYMMDDHH format
    """
    dt = pd.Timestamp(start_date)
    return [("X_" + (dt + pd.Timedelta(hours=1 * i)).strftime('%Y%m%d%H') + ".pt") for i in range(n)]

tensor_names_2018 = generate_datetime_strings(start_date_2018, n_2018)
tensor_names_2019 = generate_datetime_strings(start_date_2019, n_2019)
tensor_names_2020 = generate_datetime_strings(start_date_2020, n_2020)
tensor_names_2021 = generate_datetime_strings(start_date_2021, n_2021)

file_names = tensor_names_2018 + tensor_names_2019 + tensor_names_2020 + tensor_names_2021
print(file_names[:10])
print(len(file_names))

['X_2018071300.pt', 'X_2018071301.pt', 'X_2018071302.pt', 'X_2018071303.pt', 'X_2018071304.pt', 'X_2018071305.pt', 'X_2018071306.pt', 'X_2018071307.pt', 'X_2018071308.pt', 'X_2018071309.pt']
26280


In [ ]:
from torch.utils.data import Dataset, DataLoader

class WeatherDataset(Dataset):
    def __init__(self, file_names, targets_dict, metadata, lead_time=24):
        self.file_names = file_names
        # Need to make sure these keys exist!!!!!!!!!!!!!!!
        self.target_values = targets_dict['values']       # The 5 regression vars
        self.target_labels = targets_dict['binary_label'] # The 1 rain label
        self.lead_time = lead_time
        
        # Crop parameters (optional but recommended)
        self.iy = metadata['jumbo_y_idx']
        self.ix = metadata['jumbo_x_idx']

    def __len__(self):
        # We stop 24 hours early because we don't have "future" targets for the last day
        return len(self.file_names) - self.lead_time

    def __getitem__(self, idx):
        # 1. Load Input (Current Time)
        fname = self.file_names[idx]
        # Extract year from filename to find the right folder
        year = fname.split('_')[1][:4] 
        path = f"{DATASET_DIR}/inputs/{year}/{fname}"
        
        x = torch.load(path, weights_only=True).float() # (450, 449, 42)
        x = x.permute(2, 0, 1) # PyTorch expects (Channels, H, W)
        
        # 2. Load Target (Current Time + 24 Hours)
        target_idx = idx + self.lead_time
        y_reg = self.target_values[target_idx]
        y_cls = self.target_labels[target_idx]
        
        return x, y_reg, y_cls

In [ ]:
# Model architecture
class WeatherCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = ConvLayer(42, 64)
        self.pool1  = MaxPoolLayer()
        self.layer2 = ConvLayer(64, 128)
        self.pool2  = MaxPoolLayer()
        # You can add more layers here to get deeper!
        
        self.head   = FCLayer() 

    def forward(self, x):
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2(x)
        x = self.pool2(x)
        
        out = self.head(x) # Returns 7 values
        return out

In [ ]:
from collections import defaultdict

print("Auditing NaN channels across dataset...")
nan_channel_counts = defaultdict(int)
nan_files = []

for fname in tensor_names_2018[:500]:  # sample first 500 files
    year = fname.split('_')[1][:4]
    path = f"{DATASET_DIR}/inputs/{year}/{fname}"
    x = torch.load(path, weights_only=True).float()
    x = x.permute(2, 0, 1)  # (42, H, W)
    
    if x.isnan().any():
        nan_channels = [c for c in range(x.shape[0]) if x[c].isnan().any()]
        nan_files.append((fname, nan_channels))
        for c in nan_channels:
            nan_channel_counts[c] += 1

print(f"\nFiles with NaN: {len(nan_files)} / 500")
print(f"NaN by channel: {dict(sorted(nan_channel_counts.items()))}")
print(f"\nProblematic files:")
for fname, channels in nan_files:
    print(f"  {fname}  →  NaN channels: {channels}")

Auditing NaN channels across dataset...

Files with NaN: 84 / 500
NaN by channel: {5: 84}

Problematic files:
  X_2018071305.pt  →  NaN channels: [5]
  X_2018071306.pt  →  NaN channels: [5]
  X_2018071307.pt  →  NaN channels: [5]
  X_2018071308.pt  →  NaN channels: [5]
  X_2018071405.pt  →  NaN channels: [5]
  X_2018071406.pt  →  NaN channels: [5]
  X_2018071407.pt  →  NaN channels: [5]
  X_2018071408.pt  →  NaN channels: [5]
  X_2018071505.pt  →  NaN channels: [5]
  X_2018071506.pt  →  NaN channels: [5]
  X_2018071507.pt  →  NaN channels: [5]
  X_2018071508.pt  →  NaN channels: [5]
  X_2018071605.pt  →  NaN channels: [5]
  X_2018071606.pt  →  NaN channels: [5]
  X_2018071607.pt  →  NaN channels: [5]
  X_2018071608.pt  →  NaN channels: [5]
  X_2018071705.pt  →  NaN channels: [5]
  X_2018071706.pt  →  NaN channels: [5]
  X_2018071707.pt  →  NaN channels: [5]
  X_2018071708.pt  →  NaN channels: [5]
  X_2018071805.pt  →  NaN channels: [5]
  X_2018071806.pt  →  NaN channels: [5]
  X_201807

In [ ]:
# # 1. Setup
# print("Setup")
# dataset = WeatherDataset(tensor_names_2018, targets, metadata)
# train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
# # device = torch.device("cuda")
# device = torch.device("cpu")

# model = WeatherCNN().to(device)
# optimizer = optim.AdamW(model.parameters(), lr=1e-3)
# mse_loss_fn = nn.MSELoss()
# bce_loss_fn = nn.BCEWithLogitsLoss()

# print("Begining training")

# # 2. Loop
# for epoch in range(10):
#     print("Starting epoch", epoch)
#     for batch_x, batch_y_reg, batch_y_cls in train_loader:
#         batch_x = batch_x.to(device)
        
#         # Forward
#         preds = model(batch_x) # (B, 6)
        
#         # Calculate Loss
#         loss_reg = mse_loss_fn(preds[:, :6], batch_y_reg)
#         loss_cls = bce_loss_fn(preds[:, 6], batch_y_cls.float())
#         total_loss = loss_reg + loss_cls
        
#         # Backward
#         optimizer.zero_grad()
#         total_loss.backward()
#         optimizer.step()
        
#     print(f"Epoch {epoch} complete. Loss: {total_loss.item()}")